In [1]:
#To check whether the packages are loaded
import sys
import pandas as pd
import sqlalchemy

print(f"Python: {sys.executable}")
print(f"Pandas: {pd.__version__}")
print(f"SQLAlchemy: {sqlalchemy.__version__}")

Python: c:\Users\ferdi\miniconda3\envs\rqad\python.exe
Pandas: 3.0.3
SQLAlchemy: 2.0.50


In [3]:
#To connect to MySQL
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")

connection_url = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

#small test to try connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT VERSION();"))
    version = result.scalar()
    print(f"Yeah working {version}")


Yeah working 8.0.46


In [4]:
#to show tables in the TWOS database
tables_df = pd.read_sql("SHOW TABLES;" , engine)
tables_df

,Tables_in_tawos
0,affected_version
1,change_log
2,comment
3,component
4,fix_version
5,issue
6,issue_component
7,issue_link
8,project
9,repository


In [5]:
# show issue table
schema_query = """
SELECT
    COLUMN_NAME,
    DATA_TYPE,
    IS_NULLABLE,
    COLUMN_KEY
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'TAWOS'
AND TABLE_NAME = 'issue'
ORDER BY ORDINAL_POSITION;
"""
issue_schema =pd.read_sql(schema_query, engine)
print(f"Issue table - {len(issue_schema)} colum")
issue_schema

Issue table - 30 colum


,COLUMN_NAME,DATA_TYPE,IS_NULLABLE,COLUMN_KEY
0,ID,int,NO,PRI
1,Jira_ID,int,YES,
2,Issue_Key,varchar,YES,
3,URL,varchar,YES,
4,Title,varchar,YES,
5,Description,mediumtext,YES,
6,Description_Text,mediumtext,YES,
7,Description_Code,mediumtext,YES,
8,Type,varchar,YES,
9,Priority,varchar,YES,


In [6]:
#show first 10 issues
sample_query = """
SELECT 
    ID,
    Issue_Key,
    Title,
    Description_Text,
    Type,
    Priority,
    Status,
    Resolution,
    Story_Point,
    Project_ID,
    Creation_Date,
    Resolution_Time_Minutes
FROM issue
LIMIT 10;
"""

sample_df = pd.read_sql(sample_query, engine)
print(f"Shape: {sample_df.shape}")
sample_df

Shape: (10, 12)


,ID,Issue_Key,Title,Description_Text,Type,Priority,Status,Resolution,Story_Point,Project_ID,Creation_Date,Resolution_Time_Minutes
0,65,XD-3768,"""How do I make a job restartable in spring xd""","""""""The jobs that appear under Executions secti...",Bug,Major,To Do,NaN,1.0,1,2017-07-10 13:41:25,0.0
1,66,XD-3767,"""admin config timezone command does not work""","""""""Working with Spring-XD version 1.3.2.RELEAS...",Bug,Trivial,To Do,NaN,1.0,1,2017-06-26 16:26:27,0.0
2,67,XD-3766,"""Module Upload command not pushing jar to all ...","""""""My project 7 node cluster and in that 2 nod...",Bug,Major,To Do,NaN,10.0,1,2017-05-19 22:28:43,0.0
3,68,XD-3765,"""Fix stream failover ""","""""""See https://github.com/spring-projects/spri...",Story,Minor,Done,Complete,8.0,1,2017-03-21 16:54:44,0.0
4,69,XD-3764,"""SpringXD Job is still executing even after fo...","""""""I'm trying to run a Job on SpringXD and the...",Bug,Major,To Do,NaN,5.0,1,2017-03-06 20:01:41,0.0
5,70,XD-3763,"""Duplicate entries in table XD_JOB_REGISTRY""","""""""Hello, I have encountered following proble...",Bug,Major,To Do,NaN,1.0,1,2017-01-23 10:21:08,0.0
6,71,XD-3762,"""Spring XD Dynamic Module ClassLoader Issue""","""""""According to the documentation we can load ...",Bug,Critical,To Do,NaN,1.0,1,2016-11-07 09:10:20,0.0
7,72,XD-3761,"""java.util.concurrent.RejectedExecutionExcepti...","""""""Hi, We use spring config server 1.0.4-REL...",Bug,Major,To Do,NaN,1.0,1,2016-09-25 09:03:53,0.0
8,73,XD-3760,"""SpringXD authenication and authorization usin...","""""""Hi, I'm using LDAP to a Windows Active Dir...",Improvement,Minor,To Do,NaN,1.0,1,2016-09-19 06:43:48,0.0
9,74,XD-3759,"""Composed job endpoint is missing from the def...","""""""More details in the support ticket: https:/...",Bug,Minor,To Do,NaN,1.0,1,2016-07-19 21:56:06,0.0


In [7]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       10 non-null     int64         
 1   Issue_Key                10 non-null     str           
 2   Title                    10 non-null     str           
 3   Description_Text         10 non-null     str           
 4   Type                     10 non-null     str           
 5   Priority                 10 non-null     str           
 6   Status                   10 non-null     str           
 7   Resolution               1 non-null      str           
 8   Story_Point              10 non-null     float64       
 9   Project_ID               10 non-null     int64         
 10  Creation_Date            10 non-null     datetime64[us]
 11  Resolution_Time_Minutes  10 non-null     float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(7)

In [8]:
totals_query = """
SELECT 
    (SELECT COUNT(*) FROM issue) AS total_issues,
    (SELECT COUNT(*) FROM project) AS total_projects,
    (SELECT COUNT(DISTINCT Type) FROM issue) AS unique_types,
    (SELECT COUNT(DISTINCT Status) FROM issue) AS unique_statuses;
"""

totals = pd.read_sql(totals_query, engine)
totals

,total_issues,total_projects,unique_types,unique_statuses
0,458232,39,24,75


In [9]:
type_distribution_query = """
SELECT 
    Type, 
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM issue), 2) AS percentage
FROM issue
GROUP BY Type
ORDER BY count DESC;
"""

type_df = pd.read_sql(type_distribution_query, engine)
type_df

,Type,count,percentage
0,Bug,215570,47.04
1,Suggestion,96370,21.03
2,Improvement,42691,9.32
3,Story,31394,6.85
4,Task,28338,6.18
5,Sub-task,14396,3.14
6,New Feature,14239,3.11
7,Epic,4157,0.91
8,Enhancement Request,3239,0.71
9,Support Request,2368,0.52


In [10]:
project_dist_query = """
SELECT 
    p.Name AS project_name,
    COUNT(i.ID) AS issue_count
FROM issue i
JOIN project p ON i.Project_ID = p.ID
GROUP BY p.Name
ORDER BY issue_count DESC
LIMIT 15;
"""

project_df = pd.read_sql(project_dist_query, engine)
project_df

,project_name,issue_count
0,Moodle,66741
1,MongoDB Core Server,48663
2,Atlassian Jira Server,44165
3,Atlassian Confluence Server,42324
4,Lsstcorp Data management,26506
5,Atlassian Jira Cloud,25669
6,Atlassian Confluence Cloud,23409
7,The Titanium SDK,22059
8,Atlassian Bamboo,14252
9,Hyperledger Fabric,13682


In [11]:
#important for this project - understand story
story_by_project_query = """
SELECT 
    p.Name AS project_name,
    COUNT(i.ID) AS story_count
FROM issue i
JOIN project p ON i.Project_ID = p.ID
WHERE i.Type = 'Story'
GROUP BY p.Name
ORDER BY story_count DESC;
"""

story_by_project = pd.read_sql(story_by_project_query, engine)
print(f"{len(story_by_project)}")
print(f"{story_by_project['story_count'].sum()}")
story_by_project

22
31394


,project_name,story_count
0,Lsstcorp Data management,19578
1,Hyperledger Fabric,2748
2,Spring XD,2593
3,The Titanium SDK,1225
4,Appcelerator Studio,914
5,Apache Usergrid,805
6,Lyrasis Dura Cloud,627
7,Aptana Studio,531
8,Titanium Mobile Platform,454
9,DotNetNuke Platform,402


In [12]:
story_status_query = """
SELECT 
    Status,
    Resolution,
    COUNT(*) AS count
FROM issue
WHERE Type = 'Story'
GROUP BY Status, Resolution
ORDER BY count DESC;
"""

status_dist = pd.read_sql(story_status_query, engine)
status_dist

,Status,Resolution,count
0,Done,Done,14857
1,To Do,NaN,2917
2,Closed,Fixed,2610
3,Closed,Done,1982
4,Done,Complete,1752
...,...,...,...
78,Resolved,Later,1
79,In Progress,Done,1
80,Selected for Development,NaN,1
81,Resolved,Incomplete,1


In [13]:
# do we need cleaning
null_check_query = """
SELECT 
    COUNT(*) AS total_stories,
    SUM(CASE WHEN Title IS NULL OR Title = '' THEN 1 ELSE 0 END) AS missing_title,
    SUM(CASE WHEN Description IS NULL OR Description = '' THEN 1 ELSE 0 END) AS missing_description,
    SUM(CASE WHEN Description_Text IS NULL OR Description_Text = '' THEN 1 ELSE 0 END) AS missing_desc_text,
    SUM(CASE WHEN Story_Point IS NULL THEN 1 ELSE 0 END) AS missing_story_point,
    SUM(CASE WHEN Priority IS NULL OR Priority = '' THEN 1 ELSE 0 END) AS missing_priority
FROM issue
WHERE Type = 'Story';
"""

null_check = pd.read_sql(null_check_query, engine)
null_check.T

,0
total_stories,31394.0
missing_title,0.0
missing_description,4067.0
missing_desc_text,4067.0
missing_story_point,9661.0
missing_priority,19582.0


In [14]:
# dataFrame
main_query = """
SELECT 
    i.ID,
    i.Issue_Key,
    p.Name AS Project_Name,
    i.Title,
    i.Description,
    i.Description_Text,
    i.Type,
    i.Status,
    i.Resolution,
    i.Story_Point,
    i.Creation_Date,
    i.Resolution_Date,
    i.Resolution_Time_Minutes,
    i.Story_Point_Changed_After_Estimation
FROM issue i
JOIN project p ON i.Project_ID = p.ID
WHERE i.Type = 'Story'
  AND i.Title IS NOT NULL
  AND i.Title != '';
"""

df = pd.read_sql(main_query, engine)
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head()

Shape: (31394, 14)
Memory: 27.79 MB


,ID,Issue_Key,Project_Name,Title,Description,Description_Text,Type,Status,Resolution,Story_Point,Creation_Date,Resolution_Date,Resolution_Time_Minutes,Story_Point_Changed_After_Estimation
0,68,XD-3765,Spring XD,"""Fix stream failover ""","""See https://github.com/spring-projects/spring...","""""""See https://github.com/spring-projects/spri...",Story,Done,Complete,8.0,2017-03-21 16:54:44,2017-03-21 16:55:23,0.0,0
1,77,XD-3756,Spring XD,"""wordcount failed to run in cloudera VM 5.7""","""I download the spring XD example projects, an...","""""""I download the spring XD example projects, ...",Story,To Do,NaN,3.0,2016-05-23 10:11:59,NaT,0.0,0
2,109,XD-3724,Spring XD,"""Add Job RDBMS config in Ambari plugin""","""Spring XD Ambari plugin only supports HDB as ...","""""""Spring XD Ambari plugin only supports HDB a...",Story,To Do,NaN,2.0,2015-12-16 18:38:51,NaT,0.0,0
3,118,XD-3715,Spring XD,"""Move k8s SPI to a separate repo""","""As a developer, I'd like to move k8s SPI to i...","""""""As a developer, I'd like to move k8s SPI to...",Story,Done,Complete,5.0,2015-11-30 15:11:45,2015-12-08 20:25:41,11833.0,0
4,119,XD-3714,Spring XD,"""Upgrade XD Ambari release to 1.3 ""","""As a developer, I'd like to upgrade Spring XD...","""""""As a developer, I'd like to upgrade Spring ...",Story,Done,Complete,3.0,2015-11-30 14:33:01,2015-12-02 16:39:59,3006.0,0


In [15]:
#general
print("=== INFO ===")
df.info()
print("\n=== DESCRIPTIVE STATISTICS ===")
df.describe()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 31394 entries, 0 to 31393
Data columns (total 14 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   ID                                    31394 non-null  int64         
 1   Issue_Key                             31394 non-null  str           
 2   Project_Name                          31394 non-null  str           
 3   Title                                 31394 non-null  str           
 4   Description                           31394 non-null  str           
 5   Description_Text                      31394 non-null  str           
 6   Type                                  31394 non-null  str           
 7   Status                                31394 non-null  str           
 8   Resolution                            26540 non-null  str           
 9   Story_Point                           21733 non-null  float64       
 

,ID,Story_Point,Creation_Date,Resolution_Date,Resolution_Time_Minutes,Story_Point_Changed_After_Estimation
count,31394.000000,21733.000000,31394,26533,3.139400e+04,31394.000000
mean,211776.718927,4.352443,2017-01-30 11:39:54.742785,2017-05-22 21:12:54.823578,1.700838e+05,0.131745
min,68.000000,0.000000,2009-08-20 00:18:26,2010-01-04 16:57:15,0.000000e+00,0.000000
25%,219818.250000,1.000000,2015-05-28 12:23:35,2015-09-04 11:48:39,3.852500e+02,0.000000
50%,230668.500000,3.000000,2017-05-09 22:35:39,2017-09-22 02:33:19,2.054200e+04,0.000000
75%,241162.750000,5.000000,2019-02-14 18:18:26.750000,2019-06-04 21:33:24,1.196928e+05,0.000000
max,460229.000000,105.000000,2020-10-22 16:40:10,2020-10-22 17:35:28,4.330196e+06,1.000000
std,101790.824102,5.404201,NaN,NaN,4.102984e+05,0.338219


In [16]:
#save the dataframe as a CSV file
output_path = "../data/interim/01_user_stories_raw.csv"
df.to_csv(output_path, index=False, encoding='utf-8')

print(f"{output_path}")
print(f"~{df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

../data/interim/01_user_stories_raw.csv
~28 MB


In [17]:
df['title_char_count'] = df['Title'].str.len()
df['title_word_count'] = df['Title'].str.split().str.len()

print("TITLE Character Count")
print(df['title_char_count'].describe())

print("\nTITLE Word Count")
print(df['title_word_count'].describe())

print("\n Shortest")
print(df.nsmallest(3, 'title_char_count')[['Issue_Key', 'Project_Name', 'Title', 'title_char_count']])

print("\n Longest")
print(df.nlargest(3, 'title_char_count')[['Issue_Key', 'Project_Name', 'Title', 'title_char_count']])

TITLE Character Count
count    31394.000000
mean        50.820252
std         20.976943
min          4.000000
25%         36.000000
50%         48.000000
75%         62.000000
max        254.000000
Name: title_char_count, dtype: float64

TITLE Word Count
count    31394.000000
mean         7.300153
std          3.507539
min          1.000000
25%          5.000000
50%          7.000000
75%          9.000000
max         44.000000
Name: title_word_count, dtype: float64

 Shortest
        Issue_Key   Project_Name  Title  title_char_count
4195  APSTUD-7795  Aptana Studio   "OK"                 4
4163  APSTUD-8070  Aptana Studio  "jdk"                 5
4235  APSTUD-7432  Aptana Studio  "FTP"                 5

 Longest
         Issue_Key         Project_Name  \
8231       FAB-195   Hyperledger Fabric   
29375  TISTUD-2392  Appcelerator Studio   
8232       FAB-194   Hyperledger Fabric   

                                                   Title  title_char_count  
8231   "Ledger cross-peer c

In [18]:
#Description length features
df['desc_char_count'] = df['Description_Text'].fillna('').str.len()
df['desc_word_count'] = df['Description_Text'].fillna('').str.split().str.len()
df['has_description'] = df['desc_char_count'] > 0

#Lowercase a single time
desc_lower = df['Description_Text'].fillna('').str.lower()

# Cohn template part 1: ROLE marker
df['has_as_a'] = desc_lower.str.contains(r'\bas an?\b', regex=True, na=False)

#Cohn template part 2: MEANS marker
df['has_i_want'] = desc_lower.str.contains(
    r'\bi (want|need|would like|can|should|am able)\b', regex=True, na=False
)

#Cohn template part 3: ENDS marker (optional but recommended)
df['has_so_that'] = desc_lower.str.contains(r'\bso that\b', regex=True, na=False)

#Combined format flags
df['well_formed_min']  = df['has_as_a'] & df['has_i_want']

# Full Cohn template: role + means + reason
df['cohn_full_format'] = df['has_as_a'] & df['has_i_want'] & df['has_so_that']

print("DESCRIPTION LENGTH")
print(df['desc_char_count'].describe().round(1))
print()
print("DESCRIPTION WORD COUNT")
print(df['desc_word_count'].describe().round(1))
print()

print("USER STORY FORMAT PATTERNS")
total = len(df)

def pct(series):
    return f"{series.sum():>6,} ({series.mean()*100:5.1f}%)"

print(f"Has description (non-empty):           {pct(df['has_description'])}")
print(f"Has 'as a / as an' (role marker):      {pct(df['has_as_a'])}")
print(f"Has 'I want / need / ...' (means):     {pct(df['has_i_want'])}")
print(f"Has 'so that ...' (reason marker):     {pct(df['has_so_that'])}")
print()
print(f"QUS Well-formed (role + means):        {pct(df['well_formed_min'])}")
print(f"Full Cohn template (role+means+reason): {pct(df['cohn_full_format'])}")

DESCRIPTION LENGTH
count     31394.0
mean        333.9
std         991.6
min           0.0
25%          94.0
50%         202.0
75%         397.0
max      120122.0
Name: desc_char_count, dtype: float64

DESCRIPTION WORD COUNT
count    31394.0
mean        47.5
std         72.9
min          0.0
25%         13.0
50%         30.0
75%         60.0
max       4381.0
Name: desc_word_count, dtype: float64

USER STORY FORMAT PATTERNS
Has description (non-empty):           27,327 ( 87.0%)
Has 'as a / as an' (role marker):       2,390 (  7.6%)
Has 'I want / need / ...' (means):      1,381 (  4.4%)
Has 'so that ...' (reason marker):      1,235 (  3.9%)

QUS Well-formed (role + means):           871 (  2.8%)
Full Cohn template (role+means+reason):    284 (  0.9%)


C:\Users\ferdi\AppData\Local\Temp\ipykernel_32124\3337329170.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['has_i_want'] = desc_lower.str.contains(


In [19]:
title_lower = df['Title'].fillna('').str.lower()

df['title_has_as_a']    = title_lower.str.contains(r'\bas an?\b', regex=True, na=False)
df['title_has_i_want']  = title_lower.str.contains(r'\bi (?:want|need|would like|can|should|am able)\b', regex=True, na=False)
df['title_has_so_that'] = title_lower.str.contains(r'\bso that\b', regex=True, na=False)

print("FORMAT MARKERS IN TITLE")
print(f"Title has 'as a':       {df['title_has_as_a'].sum():>5,} ({df['title_has_as_a'].mean()*100:.1f}%)")
print(f"Title has 'I want':     {df['title_has_i_want'].sum():>5,} ({df['title_has_i_want'].mean()*100:.1f}%)")
print(f"Title has 'so that':    {df['title_has_so_that'].sum():>5,} ({df['title_has_so_that'].mean()*100:.1f}%)")

df['has_as_a_anywhere']    = df['has_as_a'] | df['title_has_as_a']
df['has_i_want_anywhere']  = df['has_i_want'] | df['title_has_i_want']
df['has_so_that_anywhere'] = df['has_so_that'] | df['title_has_so_that']

df['cohn_anywhere'] = df['has_as_a_anywhere'] & df['has_i_want_anywhere'] & df['has_so_that_anywhere']
print()
print(f"Full Cohn template ANYWHERE (title or desc): {df['cohn_anywhere'].sum():>5,} ({df['cohn_anywhere'].mean()*100:.1f}%)")

FORMAT MARKERS IN TITLE
Title has 'as a':         756 (2.4%)
Title has 'I want':       650 (2.1%)
Title has 'so that':       60 (0.2%)

Full Cohn template ANYWHERE (title or desc):   321 (1.0%)


In [20]:
sp = df['Story_Point']
n_total = len(sp)
n_missing = sp.isna().sum()
n_present = n_total - n_missing

print("STORY POINT COVERAGE")
print(f"Total stories:          {n_total:>6,}")
print(f"With Story_Point:       {n_present:>6,} ({n_present/n_total*100:.1f}%)")
print(f"Missing Story_Point:    {n_missing:>6,} ({n_missing/n_total*100:.1f}%)")
print()

print("STORY POINT — DISTRIBUTION")
print(sp.dropna().describe().round(2))
print()

# Fibonacci-like values
print("STORY POINT — VALUE FREQUENCY")
print(sp.dropna().astype(float).value_counts().head(15))
print()

# Outliers
print("STORY POINT — POTENTIAL OUTLIERS")
outliers = df[df['Story_Point'] >= 40][
    ['Issue_Key', 'Project_Name', 'Title', 'Story_Point', 'Status']
].sort_values('Story_Point', ascending=False)
print(f"Stories with SP ≥ 40: {len(outliers)}")
outliers.head(10)

STORY POINT COVERAGE
Total stories:          31,394
With Story_Point:       21,733 (69.2%)
Missing Story_Point:     9,661 (30.8%)

STORY POINT — DISTRIBUTION
count    21733.00
mean         4.35
std          5.40
min          0.00
25%          1.00
50%          3.00
75%          5.00
max        105.00
Name: Story_Point, dtype: float64

STORY POINT — VALUE FREQUENCY
Story_Point
1.00     4165
2.00     3648
3.00     2285
4.00     2082
5.00     1643
8.00     1640
6.00     1059
10.00     910
0.50      718
0.00      583
12.00     258
20.00     206
13.00     205
7.00      182
0.25      157
Name: count, dtype: int64

STORY POINT — POTENTIAL OUTLIERS
Stories with SP ≥ 40: 75


,Issue_Key,Project_Name,Title,Story_Point,Status
25815,DM-3683,Lsstcorp Data management,"""Jira Data Management long range planning proj...",105.00,Done
25816,DM-3682,Lsstcorp Data management,"""LDM-240 Long range planning""",105.00,Done
100,XD-3562,Spring XD,"""Be aware of exceptions thrown in a stream""",100.00,To Do
21702,DM-9372,Lsstcorp Data management,"""Load WISE catalog data in PDAC""",100.00,Done
27770,DM-271,Lsstcorp Data management,"""Setup the new Buildbot CI system""",100.00,Done
27104,DM-1447,Lsstcorp Data management,"""Improve spatial-selection flexibility by pars...",100.00,To Do
4728,DAEMON-203,Appcelerator Daemon,"""Create a feature film starring the Appc Daemon""",99.00,Resolved
4568,APSTUD-2241,Aptana Studio,"""Open Declaration / Selection in JavaScript""",89.00,Closed
4375,APSTUD-4186,Aptana Studio,"""When indexing ruby gem loadpath, limit to ind...",89.00,Open
14361,DM-19376,Lsstcorp Data management,"""FY19 Received Equipment""",88.76,Done
